In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
%pip install -U dagshub mlflow skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 79.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import dagshub
dagshub.init(repo_owner='gvakh23', repo_name='ML_assignment2', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=856d781e-3406-4119-881b-f2e88ba2c7f4&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=31e72eab2b22986e6445a120d69c177ed775685da9501f2b5a912cf3bb085feb




Output()

Accessing as gvakh23

Initialized MLflow to track repo "gvakh23/ML_assignment2"

Repository gvakh23/ML_assignment2 initialized!

In [5]:
import mlflow
print(mlflow.get_tracking_uri())

https://dagshub.com/gvakh23/ML_assignment2.mlflow


In [6]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

train_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_ident  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = train_trans.merge(train_ident, on='TransactionID', how='left')


In [7]:
train_sorted = train.sort_values('TransactionDT')

val_size = int(len(train_sorted) * 0.2)
train_df = train_sorted.iloc[:-val_size].copy()
val_df   = train_sorted.iloc[-val_size:].copy()

print(f"Train split : {train_df.shape}, fraud rate: {train_df['isFraud'].mean():.2%}")
print(f"Val   split : {val_df.shape},   fraud rate: {val_df['isFraud'].mean():.2%}")


Train split : (472432, 434), fraud rate: 3.51%
Val   split : (118108, 434),   fraud rate: 3.44%


## Cleaning

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
class Cleaner(BaseEstimator, TransformerMixin):
    """
    Drops high-null columns, encodes M columns (T/F→1/0),
    fills remaining NaN with -999.
    All thresholds computed on train only.
    """
 
    def __init__(self, null_threshold=0.9, fill_value=-999):
        self.null_threshold = null_threshold
        self.fill_value     = fill_value
 
    def fit(self, X, y=None):
        print("clean_in")

        df = X.copy()
 
        null_rates = df.isnull().mean()
        self.cols_to_drop_ = null_rates[null_rates > self.null_threshold].index.tolist()
        self.cols_to_drop_ = [c for c in self.cols_to_drop_
                               if c not in ['isFraud', 'TransactionID']]
 
        # Learn M column names
        self.m_cols_ = [c for c in df.columns if c.startswith('M')]
 
        with mlflow.start_run(run_name='LogisticRegression_Cleaning', nested=True):
            mlflow.log_param('null_threshold',  self.null_threshold)
            mlflow.log_param('fill_value',      self.fill_value)
            mlflow.log_param('fill_reason',     'tree_model_out_of_range_sentinel')
            mlflow.log_param('m_col_encoding',  'T=1_F=0_NaN=fill_value')
            mlflow.log_metric('cols_dropped',   len(self.cols_to_drop_))
            mlflow.log_metric('cols_remaining', df.shape[1] - len(self.cols_to_drop_))
            mlflow.log_metric('rows',           df.shape[0])
            mlflow.log_metric('fraud_rate',     float(y.mean()) if y is not None else -1)
        print(f"[Cleaner] Done — dropped {len(self.cols_to_drop_)} cols, {df.shape[0]} rows")
        return self
 
    def transform(self, X):
        print("clean trans in")
        df = X.copy()
 
        # Drop high-null cols
        df.drop(columns=[c for c in self.cols_to_drop_ if c in df.columns],
                inplace=True, errors='ignore')
 
        # M columns T/F → 1/0
        for col in self.m_cols_:
            if col in df.columns:
                df[col] = df[col].map({'T': 1, 'F': 0})
 
        # Fill NaN
        df.fillna(self.fill_value, inplace=True)
 
        return df

## Feature Engineering

In [16]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.d_cols      = ['D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D15']
        self.freq_cols   = ['card1', 'card2', 'addr1', 'uid', 'uid2', 'P_emaildomain']
        self.agg_cols    = ['card1', 'uid']          # removed card2/addr1 — less useful
        self.agg_targets = ['TransactionAmt', 'D1', 'C1', 'C2', 'C13']

    def fit(self, X, y=None):
        df = X.copy()

        # 1. D normalization mins
        self.d_min_map_ = {}
        for d in self.d_cols:
            if d in df.columns:
                self.d_min_map_[d] = df.groupby('card1')[d].min()

        # 2. Build UIDs
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        # uid2: use D1 only if it's not -999 (missing)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # 3. Frequency maps
        self.freq_maps_ = {}
        for col in self.freq_cols:
            if col in df.columns:
                self.freq_maps_[col] = df[col].value_counts()

        # 4. Aggregation maps — store as dict of Series for fast map()
        # Using dict of Series is faster than storing DataFrame and accessing columns
        self.agg_maps_ = {}
        for col in self.agg_cols:
            if col not in df.columns:
                continue
            self.agg_maps_[col] = {}
            for target in self.agg_targets:
                if target not in df.columns:
                    continue
                self.agg_maps_[col][target] = {}
                grp = df.groupby(col)[target]
                self.agg_maps_[col][target]['mean'] = grp.mean()
                self.agg_maps_[col][target]['std']  = grp.std()
                self.agg_maps_[col][target]['max']  = grp.max()

        # 5. UID label encoders — store as plain dict for fastest mapping
        self.uid_maps_ = {}
        for col in ['uid', 'uid2']:
            if col in df.columns:
                # dict map is 3-4x faster than LabelEncoder.transform in a loop
                unique_vals = df[col].astype(str).unique()
                self.uid_maps_[col] = {v: i for i, v in enumerate(unique_vals)}

        # 6. Email suffix encoder
        if 'P_emaildomain' in df.columns:
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            unique_sfx = suffixes.unique()
            self.email_suffix_map_ = {v: i for i, v in enumerate(unique_sfx)}
        else:
            self.email_suffix_map_ = {}

        with mlflow.start_run(run_name='LogisticRegression_Feature_Engineering', nested=True):
            mlflow.log_param('time_features',      'hour, day')
            mlflow.log_param('log_transform',      'TransactionAmt_log, TransactionAmt_decimal')
            mlflow.log_param('d_cols_normalized',  str([d for d in self.d_cols if d in self.d_min_map_]))
            mlflow.log_param('d_norm_method',      'subtract_card1_group_min_train_only')
            mlflow.log_param('uid',                'card1+card2')
            mlflow.log_param('uid2',               'card1+addr1+D1 (NA if D1 missing)')
            mlflow.log_param('freq_cols',          str(self.freq_cols))
            mlflow.log_param('agg_cols',           str(self.agg_cols))
            mlflow.log_param('agg_targets',        str(self.agg_targets))
            mlflow.log_param('email_features',     'same_email, P_email_suffix')
            mlflow.log_metric('d_cols_fitted',     len(self.d_min_map_))
            mlflow.log_metric('freq_cols_fitted',  len(self.freq_maps_))
            mlflow.log_metric('uid_maps_fitted',   len(self.uid_maps_))

        return self

    def transform(self, X):
        df = X.copy()

        # ── Time features ─────────────────────────────────────
        df['hour']  = (df['TransactionDT'] / 3600) % 24
        df['day']   = (df['TransactionDT'] / (3600 * 24)) % 7

        # ── Log transform ─────────────────────────────────────
        df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

        # ── Email features ────────────────────────────────────
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['same_email'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
            # Encode suffix using saved map (fast dict lookup, no lambda)
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            df['P_email_suffix'] = suffixes.map(self.email_suffix_map_).fillna(-1).astype(int)

        # ── D normalization ───────────────────────────────────
        for d, min_map in self.d_min_map_.items():
            if d in df.columns:
                df[f'{d}_norm'] = df[d] - df['card1'].map(min_map).fillna(0)

        # ── UIDs ──────────────────────────────────────────────
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # ── Frequency encoding ────────────────────────────────
        for col, freq_map in self.freq_maps_.items():
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(freq_map).fillna(0)

        # ── Aggregations (fast: map Series directly) ──────────
        for col, targets in self.agg_maps_.items():
            if col not in df.columns:
                continue
            for target, stats in targets.items():
                for stat, series in stats.items():
                    df[f'{col}_{target}_{stat}'] = df[col].map(series).fillna(-999)

        # ── UID encoding (fast: plain dict map) ───────────────
        for col, uid_map in self.uid_maps_.items():
            if col in df.columns:
                df[col] = df[col].astype(str).map(uid_map).fillna(-1).astype(int)

        return df




##  Categorical Encoding

In [10]:
import pandas as pd
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, ohe_threshold=10):
        self.ohe_threshold = ohe_threshold
        self.device_map_ = {'desktop': 1, 'mobile': 0}
        self.ohe_info_ = {}
        self.le_encoders_ = {}
        self.le_cols_ = []

    def fit(self, X, y=None):
        print("encode_in")

        df = X.copy()
        
        # 1. Identify Categorical Columns (excluding IDs/Targets)
        all_cat_cols = [c for c in df.select_dtypes(include='object').columns 
                        if c not in ['TransactionID', 'isFraud', 'DeviceType']]

        # 2. Split logic: OHE vs Label Encode based on threshold
        for col in all_cat_cols:
            n_unique = df[col].nunique()
            if n_unique < self.ohe_threshold:
                self.ohe_info_[col] = df[col].unique().tolist()
            else:
                le = LabelEncoder()
                le.fit(df[col].astype(str))
                self.le_encoders_[col] = le
                self.le_cols_.append(col)

        with mlflow.start_run(run_name='LogisticRegression_Categorical_Encoding', nested=True):
            mlflow.log_param('ohe_threshold', self.ohe_threshold)
            mlflow.log_param('ohe_cols', list(self.ohe_info_.keys()))
            mlflow.log_param('le_cols', self.le_cols_)
            mlflow.log_metric('ohe_count', len(self.ohe_info_))
            mlflow.log_metric('le_count', len(self.le_cols_))
        print(f"[CategoricalEncoder] Done — OHE cols: {len(self.ohe_info_)}, LE cols: {len(self.le_cols_)}")

        return self

    def transform(self, X):
        print("encode trans in")

        df = X.copy()

        # ── BINARY: DeviceType ──
        if 'DeviceType' in df.columns:
            df['DeviceType'] = df['DeviceType'].map(self.device_map_).fillna(-999)

        # ── ONE-HOT ENCODING ──
        for col, categories in self.ohe_info_.items():
            if col in df.columns:
        
                dummies = pd.get_dummies(df[col], prefix=col)
        
                expected_cols = [f"{col}_{cat}" for cat in categories]
                dummies = dummies.reindex(columns=expected_cols, fill_value=0)
        
                df = pd.concat([df, dummies], axis=1)
                df.drop(columns=[col], inplace=True)
                
        # ── LABEL ENCODING ──
        for col, le in self.le_encoders_.items():
            if col in df.columns:
                known_classes = set(le.classes_)
                mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
                df[col] = df[col].astype(str).map(mapping).fillna(-1)

        return df

## Feature Selection

In [11]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, variance_threshold=0.01):
        self.variance_threshold = variance_threshold
        self.drop_cols_ = []
        self.feature_cols_ = []

    def fit(self, X, y=None):
        print("select_in")

        df = X.copy()
        numeric_df = df.select_dtypes(include=[np.number])
        
        # ── METHOD 1: Variance Threshold ──
        vt = VarianceThreshold(threshold=self.variance_threshold)
        vt.fit(numeric_df)
        low_var_cols = numeric_df.columns[~vt.get_support()].tolist()

        # ── METHOD 2: V-Column NaN-Group Selection ──
        # In this dataset, V columns are often redundant. 
        # Group them by their 'NaN pattern' and keep only the one strongest with target.
        v_cols = [c for c in df.columns if c.startswith('V')]
        
        null_patterns = {}
        for col in v_cols:
            pattern = int((df[col] == -999).sum())
            null_patterns.setdefault(pattern, []).append(col)

        best_per_group = []
        if y is not None:
            y_s = pd.Series(y.values if hasattr(y, 'values') else np.array(y), index=df.index)
            for pattern, cols in null_patterns.items():
                corrs = df[cols].corrwith(y_s).abs().fillna(0)
                best_per_group.append(corrs.idxmax())
        else:
            for pattern, cols in null_patterns.items():
                best_per_group.append(cols[0])

        v_to_drop = [c for c in v_cols if c not in best_per_group]
        
        # Combine all columns to remove
        self.drop_cols_ = list(set(low_var_cols + v_to_drop))
        
        # Save the "survivor" column names in order
        self.feature_cols_ = [c for c in df.columns if c not in self.drop_cols_]

        # MLflow Logging
        with mlflow.start_run(run_name='LogisticRegression_Feature_Selection', nested=True):
            mlflow.log_param('variance_threshold', self.variance_threshold)
            mlflow.log_metric('v_nan_groups', len(null_patterns))
            mlflow.log_metric('total_features_dropped', len(self.drop_cols_))
            mlflow.log_metric('final_feature_count', len(self.feature_cols_))
        print(f"[FeatureSelector] Done — dropped {len(self.drop_cols_)} features, {len(self.feature_cols_)} remaining")

        return self

    def transform(self, X):
        print("select trans in")

        df = X.copy()
        
        # 1. Drop the identified useless columns
        df.drop(columns=[c for c in self.drop_cols_ if c in df.columns], 
                inplace=True, errors='ignore')
        
        # 2. EXACT ALIGNMENT
        df = df.reindex(columns=self.feature_cols_, fill_value=0)
        
        return df

## Scaling

In [12]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

class Scaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X)
        self.feature_names_ = list(X.columns)
        return self

    def transform(self, X):
        scaled = self.scaler_.transform(X)
        return pd.DataFrame(scaled, columns=self.feature_names_, index=X.index)

In [17]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

NULL_THRESH = 0.9
OHE_THRESH  = 10
VAR_THRESH  = 0.01

import mlflow
import mlflow.sklearn

mlflow.set_experiment("LogisticRegression_Training")

preprocessing = Pipeline([
    ('cleaning',     Cleaner(null_threshold=NULL_THRESH)),
    ('feature_eng',  FeatureEngineer()),
    ('encoding',     CategoricalEncoder(ohe_threshold=OHE_THRESH)),
    ('selection',    FeatureSelector(variance_threshold=VAR_THRESH)),
    ('scaling',     Scaler()),
])



In [18]:
X_train = train_df.drop(columns=['isFraud', 'TransactionID'])
y_train = train_df['isFraud']

X_val = val_df.drop(columns=['isFraud', 'TransactionID'])
y_val = val_df['isFraud']


In [19]:
X_train = preprocessing.fit_transform(X_train, y_train)
X_val   = preprocessing.transform(X_val)

print("Preprocessing Done")

clean_in
🏃 View run LogisticRegression_Cleaning at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/1646ca4ddabf42899c65f4986fea5f57
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3
[Cleaner] Done — dropped 12 cols, 472432 rows
clean trans in
🏃 View run LogisticRegression_Feature_Engineering at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/77b5030f4bfc4e69bf4b4b4a36f4517e
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3
encode_in
🏃 View run LogisticRegression_Categorical_Encoding at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/6e8ef6db26fb4e8f9bf3fb68a82be56c
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3
[CategoricalEncoder] Done — OHE cols: 13, LE cols: 6
encode trans in
select_in
🏃 View run LogisticRegression_Feature_Selection at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experimen

## Training


In [20]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

In [22]:
with mlflow.start_run(run_name='LR_Underfitted'):
    params = {'C': 0.001, 'max_iter': 1000, 'solver': 'saga',
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'UNDERFITTED — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {train_auc-val_auc:.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


UNDERFITTED — train: 0.7592, val: 0.7467, gap: 0.0125
🏃 View run LR_Underfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/80fd353ebe324da1a835661dd42162e7
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3


In [23]:
with mlflow.start_run(run_name='LR_Overfitted'):
    params = {'C': 1000, 'max_iter': 1000, 'solver': 'saga',
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'OVERFITTED  — train: {train_auc:.4f}, val: {val_auc:.4f}, gap: {train_auc-val_auc:.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


OVERFITTED  — train: 0.7599, val: 0.7470, gap: 0.0130
🏃 View run LR_Overfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/bc3e9910d07d4da4985d1434fe3cecd0
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3


In [23]:
with mlflow.start_run(run_name='LR_Baseline') as run:
    params = {'C': 1.0, 'max_iter': 1000, 'solver': 'saga',
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('cv_mean_auc', np.mean(cv_scores))
    mlflow.log_metric('cv_std_auc',  np.std(cv_scores))
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    print(f'BASELINE    — train: {train_auc:.4f}, val: {val_auc:.4f}')
    print(f'CV          — mean: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype

BASELINE    — train: 0.7494, val: 0.7440
CV          — mean: nan, std: nan
🏃 View run LR_Baseline at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/6988dd61369949278824676000a00af9
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3


In [21]:
with mlflow.start_run(run_name='LR_Tuned') as run:
    params = {'C': 0.05, 'penalty': 'l2', 'max_iter': 2000, 'solver': 'saga',
              'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    model_best = LogisticRegression(**params)
    model_best.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, model_best.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(val_df['isFraud'], model_best.predict_proba(X_val)[:, 1])
    mlflow.log_params(params)
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    best_lr_run_id = run.info.run_id
    print(f'TUNED       — train: {train_auc:.4f}, val: {val_auc:.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


TUNED       — train: 0.8431, val: 0.8114
🏃 View run LR_Tuned at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/9412e94597204b50abb601bbf64a28ac
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3


In [22]:
with mlflow.start_run(run_name='LR_FullPipeline_Registry') as run:

    print('Building full pipeline...')
    full_pipeline = Pipeline([
        ('cleaning',    preprocessing.named_steps['cleaning']),
        ('feature_eng', preprocessing.named_steps['feature_eng']),
        ('encoding',    preprocessing.named_steps['encoding']),
        ('selection',   preprocessing.named_steps['selection']),
        ('scaling',     preprocessing.named_steps['scaling']),
        ('model',       model_best)
    ])

    # Verify it works on raw data before saving
    print('Verifying pipeline on raw sample...')
    raw_sample = train_df.iloc[:100].drop(columns=['isFraud', 'TransactionID'])
    test_preds = full_pipeline.predict_proba(raw_sample)[:, 1]
    print(f'Sample preds: min={test_preds.min():.3f}, max={test_preds.max():.3f}')
    print('Pipeline verified!')

    # Log params and metrics
    best_params = {'C': 0.05, 'penalty': 'l2', 'max_iter': 2000, 'solver': 'saga',
                   'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
    mlflow.log_params(best_params)
    mlflow.log_param('pipeline_steps', 'Cleaner→FeatureEngineer→CategoricalEncoder→FeatureSelector→Scaler→LR')
    mlflow.log_metric('val_auc', roc_auc_score(val_df['isFraud'], model_best.predict_proba(X_val)[:, 1]))

    # Save
    mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        name='full_lr_pipeline'
    )

    lr_pipeline_run_id = run.info.run_id
    print(f'\nFull pipeline saved!')
    print(f'Run ID: {lr_pipeline_run_id}')

Building full pipeline...
Verifying pipeline on raw sample...
clean trans in
encode trans in
select trans in
Sample preds: min=0.010, max=0.810
Pipeline verified!


2026/05/06 10:44:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Full pipeline saved!
Run ID: 41885b6b979f428f9d152de2cca874b1
🏃 View run LR_FullPipeline_Registry at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3/runs/41885b6b979f428f9d152de2cca874b1
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/3
